# Analyze a saved self-driving scan

Replays a `.npz` saved by `Run_SelfDriving_Scan.ipynb` -- **no hardware, no
re-scanning** -- so you can tune the reconstruction against the data you already
collected, as many times as you want, and save an **animation** of the run.

Run cells top to bottom:
1. **Load** -- picks the newest run in `scan_data/` (or set `SCAN_FILE`).
2. **Raw trajectory** -- commanded vs. sensor path.
3. **Reconstruction settings** -- choose `"compressed_sensing"` or `"gp"` (or
   `"interpolated"`) and tweak its parameters. Used by sections 4 and 5.
4. **Final reconstruction** -- binned AI0 / saved reconstruction / recomputed one.
5. **Animation** -- the run replayed scan by scan, like the live view (scan path |
   binned AI0 | reconstruction), saved as a GIF (or MP4 if ffmpeg is installed).
6. **Save** the recomputed reconstruction (optional; the original file is never modified).

In [ ]:
import pathlib
import time
import numpy as np
import matplotlib.pyplot as plt

from selfdriving_afm.config import LiveViewConfig
from selfdriving_afm.reconstruction import bin_image, reconstruct_image, inpaint_binned
from selfdriving_afm.viz import build_grid, subsample_for_reconstruction, _KIND_STYLE

## 1. Load a saved run

In [ ]:
SCAN_FILE = "scan_data/scan_edge_explorer_20261014_160337.npz"  # None = the newest .npz in scan_data/, or e.g. "scan_data/scan_edge_follow_20260923_150000.npz"

if SCAN_FILE is None:
    candidates = sorted(p for p in pathlib.Path("scan_data").glob("scan_*.npz") if "_reanalyzed" not in p.name)
    if not candidates:
        raise FileNotFoundError(f"No scan_*.npz in {pathlib.Path('scan_data').resolve()} -- set SCAN_FILE")
    SCAN_FILE = str(candidates[-1])

t_load = time.time()
d = np.load(SCAN_FILE, allow_pickle=True)
MODALITY = str(d["modality"])
HAS_SENSORS = "x_sensor" in d.files
# Z units: "nm" if the run converted Z (cfg.z_signal.units = "nm"), else AI0 volts
Z_UNIT = str(d["z_unit"]) if "z_unit" in d.files else "V"
Z_LABEL = "Z (nm)" if Z_UNIT == "nm" else "AI0 (V)"
if "z_label" in d.files:                  # e.g. PFM: "PFM phase (deg)"
    Z_LABEL = str(d["z_label"])
points = d["points"]  # (x, y, z) per sample -- sensor-corrected already if the run had sensors on
KINDS = [str(k) for k in d["kinds"]]
SEG_LENGTHS = np.asarray(d["segment_lengths"])

print(f"Loaded {SCAN_FILE} in {time.time() - t_load:.1f}s")
print(f"  modality: {MODALITY}")
print(f"  points: {len(points):,} in {len(KINDS)} waveforms "
      f"({KINDS.count('spiral')} spiral, {KINDS.count('circle')} circle, "
      f"{KINDS.count('line')} line, {KINDS.count('explore')} explorer, "
      f"{KINDS.count('grid_sparse') + KINDS.count('refine')} grid/refine, {KINDS.count('connector')} connector)")
print(f"  position sensors recorded: {HAS_SENSORS}")
if "priors_json" in d.files:
    import json as _json
    _pr = _json.loads(str(d["priors_json"]))
    _e = _pr.get("entered")
    print(f"  priors: features {_pr['feature_min_V']:.4g}..{_pr['feature_max_V']:.4g} V, area half-width "
          f"{_pr['area_half_width_V']:.4g} V at ({_pr['area_center_V'][0]:.4g}, {_pr['area_center_V'][1]:.4g}) V"
          + (f"  [entered in nm: features {_e['feature_min_nm']:g}..{_e['feature_max_nm']:g} nm, half-width "
             f"{_e['area_half_width_nm']:g} nm at ({_e['area_center_nm'][0]:g}, {_e['area_center_nm'][1]:g}) nm]"
             if _e else ""))
print(f"  Z in: {Z_UNIT}" + (f" ({float(d['z_scale_per_V']):.5g} nm per AI volt, {str(d['z_signal_json'])})" if Z_UNIT == "nm" and "z_signal_json" in d.files else ""))
print(f"  reconstruction saved at scan time: "
      f"{str(d['recon_mode']) if 'recon_mode' in d.files else 'interpolated (older run)'}")

# --- split the run back into its scans (used by the scan-area mask and the animation) ---
# * connectors (moves between scans) belong to the scan they lead into
# * a spiral/circle sent to PyScanner in several pieces (SafetyConfig.max_samples_per_waveform)
#   is ONE scan: consecutive same-kind waveforms with no move between them, or only a short
#   move back (raw readback re-measuring the samples a piece didn't record)
x_cmd, y_cmd = d["x_cmd"], d["y_cmd"]
OFFSETS = np.concatenate([[0], np.cumsum(SEG_LENGTHS)])
_names = {"spiral": "Spiral search", "circle": "Edge-trace circle", "line": "GP path segment",
          "explore": "Edge-explorer segment", "grid_sparse": "Sparse scan", "refine": "Refinement scan"}


def _fit_circle(xs, ys):
    """Least-squares (Kasa) circle fit -> (cx, cy, r); fine with a slightly short arc."""
    A = np.column_stack([xs, ys, np.ones_like(xs)])
    (a0, a1, a2), *_ = np.linalg.lstsq(A, xs ** 2 + ys ** 2, rcond=None)
    cx, cy = a0 / 2, a1 / 2
    return float(cx), float(cy), float(np.sqrt(max(a2 + cx ** 2 + cy ** 2, 0.0)))


SCANS, _pending, _counters = [], [], {}
for i, kind in enumerate(KINDS):
    if kind == "connector":
        _pending.append(i)
        continue
    if SCANS and SCANS[-1]["kind"] == kind and kind in ("spiral", "circle", "grid_sparse", "refine"):
        # a piece of the same waveform starts exactly one sample step after the last
        # RECORDED sample of the previous piece (whether or not a move back was
        # needed to get there) -- a genuinely new scan never does
        prev_last = OFFSETS[SCANS[-1]["steps"][-1] + 1] - 1
        a = OFFSETS[i]
        ds = float(np.median(np.hypot(np.diff(x_cmd[a:a + 50]), np.diff(y_cmd[a:a + 50]))))
        gap = float(np.hypot(x_cmd[a] - x_cmd[prev_last], y_cmd[a] - y_cmd[prev_last]))
        if gap <= 2.0 * ds:
            SCANS[-1]["steps"] += _pending + [i]
            SCANS[-1]["pieces"] += 1
            _pending = []
            continue
    _counters[kind] = _counters.get(kind, 0) + 1
    SCANS.append(dict(kind=kind, steps=_pending + [i], pieces=1, label=f"{_names.get(kind, kind)} #{_counters[kind]}"))
    _pending = []
if _pending:  # trailing move(s) after the last scan
    SCANS.append(dict(kind="connector", steps=_pending, pieces=1, label="Final move"))

# the disk each spiral / circle covers (for RECON_REGION = "scan_areas")
for s in SCANS:
    s["end_sample"] = int(OFFSETS[s["steps"][-1] + 1])
    own = [j for j in s["steps"] if KINDS[j] == s["kind"]]
    s["disk"] = None
    if s["kind"] not in ("spiral", "circle"):
        continue
    xs = np.concatenate([x_cmd[OFFSETS[j]:OFFSETS[j + 1]] for j in own])
    ys = np.concatenate([y_cmd[OFFSETS[j]:OFFSETS[j + 1]] for j in own])
    step = max(1, len(xs) // 20000)
    xs, ys = xs[::step], ys[::step]
    if s["kind"] == "circle":
        s["disk"] = _fit_circle(xs, ys)
    else:  # spiral: centered disk out to its outermost sample
        cx, cy = 0.5 * (xs.min() + xs.max()), 0.5 * (ys.min() + ys.max())
        s["disk"] = (float(cx), float(cy), float(np.max(np.hypot(xs - cx, ys - cy))))

print(f"  {len(SCANS)} scans: {_counters.get('spiral', 0)} spirals, {_counters.get('circle', 0)} circles, "
      f"{_counters.get('line', 0)} GP lines, {_counters.get('grid_sparse', 0)} sparse + "
      f"{_counters.get('refine', 0)} refinement scans")

## 2. Raw trajectory: commanded vs. sensor (if recorded)

If this run had `cfg.position_sensors.enabled = True`, this compares the commanded
waveform against what the position sensors actually measured -- the open-loop
discrepancy you're trying to correct for. If it didn't, this just shows the
commanded path (`x_sensor`/`y_sensor` won't exist in the file, and `points` is
identical to `x_cmd`/`y_cmd`/`z`).


In [ ]:
# a full scan can be millions of raw samples (see VelocityConfig) -- stride purely
# for plotting, same trick viz.py's live view uses; doesn't touch the loaded data
_plot_stride = max(1, len(d["x_cmd"]) // 20000)

fig, axes = plt.subplots(1, 2 if HAS_SENSORS else 1, figsize=(14, 6) if HAS_SENSORS else (7, 6))
ax0 = axes[0] if HAS_SENSORS else axes

ax0.plot(d["x_cmd"][::_plot_stride], d["y_cmd"][::_plot_stride], '-', linewidth=0.5, alpha=0.7, label="commanded")
if HAS_SENSORS:
    ax0.plot(d["x_sensor"][::_plot_stride], d["y_sensor"][::_plot_stride], '-', linewidth=0.5, alpha=0.7,
             color="tab:red", label="sensor")
ax0.set_aspect("equal")
ax0.set_xlabel("X (V)"); ax0.set_ylabel("Y (V)")
ax0.set_title("Commanded vs. sensor path" if HAS_SENSORS else "Commanded path")
ax0.legend()
ax0.grid(True, alpha=0.3)

if HAS_SENSORS:
    dx = d["x_sensor"] - d["x_cmd"]
    dy = d["y_sensor"] - d["y_cmd"]
    dr = np.sqrt(dx**2 + dy**2)
    ax1 = axes[1]
    ax1.hist(dr, bins=80)
    ax1.set_xlabel("|sensor - commanded| (V)")
    ax1.set_ylabel("count")
    ax1.set_title(f"Open-loop discrepancy\nmean={dr.mean():.5f} V, max={dr.max():.5f} V")
    ax1.grid(True, alpha=0.3)
    print(f"Open-loop discrepancy: mean={dr.mean():.5f} V, median={np.median(dr):.5f} V, max={dr.max():.5f} V")

plt.tight_layout()
plt.show()


## 3. Reconstruction settings

Used by both the final reconstruction (section 4) and the animation (section 5).
Everything starts from the **binned AI0** image -- mean AI0 per pixel, unmeasured
pixels blank, exactly like PyScanner's `image_AI0` -- and the reconstruction fills
in the blank pixels:
- `"compressed_sensing"` -- assumes the image is sparse in the 2D DCT (fast, ~0.2 s).
- `"gp"` -- Gaussian-process regression on the measured pixels (~1 s with
  `GP_OPTIMIZE = False`; much slower with it on).
- `"interpolated"` -- plain griddata interpolation, for comparison.

**Where to reconstruct** (`RECON_REGION`) -- far from anything scanned, a
reconstruction is a guess with nothing to hold it down, so by default it's only
shown where the tip actually scanned:
- `"scan_areas"` -- pixels inside a search spiral or a tracing circle (each one's
  disk, grown by `REGION_MARGIN_V`); everything else is left blank. For a
  grid+refine run: the square the sparse scan covered.
- `"near_measured"` -- pixels within `NEAR_MEASURED_V` of any measured pixel.
- `"full"` -- the whole window (the old behaviour).

In [ ]:
RECON_MODE = "compressed_sensing"               # "compressed_sensing", "gp", "linear", "fast" or "interpolated"
                                    #   "linear" = interpolation between all measured pixels -- best on steps;
                                    #   "fast" = local weighted average; "gp" = as in the SS-KPFM paper
N_PIXELS = 200                      # image size (N_PIXELS x N_PIXELS)
BOUNDS = None                       # None = same window the live view used (saved in the file),
                                    # or (xmin, xmax, ymin, ymax) in V to zoom in / out

# where to reconstruct
RECON_REGION = "full"         # "scan_areas", "near_measured" or "full"
REGION_MARGIN_V = 0.0               # "scan_areas": grow every spiral/circle disk by this much (V)
NEAR_MEASURED_V = 0.05              # "near_measured": max distance from a measured pixel (V)

# compressed sensing
CS_LAMBDA = 0.005                   # higher = smoother, lower = sticks closer to the measured pixels
CS_ITERATIONS = 500                 # more = more converged, slower
CS_DETREND = True                   # remove the sample tilt (best-fit plane) first -- sharper features

# GP
GP_KERNEL = "matern52"              # "matern52" (as in the SS-KPFM paper) or "rbf"
GP_MAX_POINTS = 3000                # measured pixels the GP trains on -- the speed knob (cost ~ n^3)
GP_LENGTH_SCALE_FRAC = 0.05         # starting feature size, as a fraction of the image width
GP_OPTIMIZE = True                  # learn length scale + noise from the data (~40 s); False = fixed, fast
                                    #   (the animation refits every RECON_EVERY frames -- raise it, or use False there)

# interpolated
INTERP_MAX_POINTS = 20000           # subsample cap before griddata

# --- the analysis window ---
if BOUNDS is None:
    if "recon_bounds" in d.files:
        BOUNDS = tuple(float(v) for v in d["recon_bounds"])
    else:  # older runs: square, padded window around everything scanned (like viz.bounds_from_history)
        _x0, _x1 = float(points[:, 0].min()), float(points[:, 0].max())
        _y0, _y1 = float(points[:, 1].min()), float(points[:, 1].max())
        _half = max(_x1 - _x0, _y1 - _y0, 1e-6) / 2 * 1.15
        _cx, _cy = (_x0 + _x1) / 2, (_y0 + _y1) / 2
        BOUNDS = (_cx - _half, _cx + _half, _cy - _half, _cy + _half)

_recon_cfg = LiveViewConfig(reconstruction=RECON_MODE, n_image_pixels=N_PIXELS, cs_lambda=CS_LAMBDA,
                            cs_iterations=CS_ITERATIONS, cs_detrend=CS_DETREND, gp_kernel=GP_KERNEL,
                            gp_max_points=GP_MAX_POINTS,
                            gp_length_scale_frac=GP_LENGTH_SCALE_FRAC, gp_optimize=GP_OPTIMIZE,
                            max_reconstruction_points=INTERP_MAX_POINTS)
RECON_TITLES = {"compressed_sensing": "CS", "gp": "GP", "linear": "Linear", "fast": "Fast inpainting",
                "interpolated": "Interpolated"}


def reconstruct_binned(binned, bounds, pts=None):
    """Fill the blank pixels of a binned image with RECON_MODE. `pts` (x, y, z rows)
    is only needed for "interpolated", which works from the raw points."""
    if RECON_MODE in ("compressed_sensing", "gp", "linear", "fast"):
        return inpaint_binned(binned, bounds, _recon_cfg)
    if RECON_MODE == "interpolated":
        return reconstruct_image(pts[:, :2], pts[:, 2], bounds, _recon_cfg)[1]
    raise ValueError(f"Unknown RECON_MODE {RECON_MODE!r}")


_GX, _GY = np.meshgrid(np.linspace(BOUNDS[0], BOUNDS[1], N_PIXELS), np.linspace(BOUNDS[2], BOUNDS[3], N_PIXELS))


def scan_disk_mask(scans):
    """Pixels inside any of these scans' spiral/circle disks (grown by REGION_MARGIN_V)."""
    mask = np.zeros(_GX.shape, dtype=bool)
    for s in scans:
        if s.get("disk") is not None:
            cx, cy, r = s["disk"]
            mask |= (_GX - cx) ** 2 + (_GY - cy) ** 2 <= (r + REGION_MARGIN_V) ** 2
    return mask


def region_mask(binned, scans):
    """Pixels to reconstruct, per RECON_REGION -- `binned` is the binned image so far,
    `scans` the scans measured so far (the animation grows the region frame by frame)."""
    measured = np.isfinite(binned)
    if RECON_REGION == "full":
        return np.ones(binned.shape, dtype=bool)
    if RECON_REGION == "scan_areas" and GRID_AREA is not None:  # grid+refine: the scanned square
        _X, _Y = np.meshgrid(np.linspace(BOUNDS[0], BOUNDS[1], binned.shape[1]),
                             np.linspace(BOUNDS[2], BOUNDS[3], binned.shape[0]))
        return ((_X >= GRID_AREA[0]) & (_X <= GRID_AREA[1]) & (_Y >= GRID_AREA[2]) & (_Y <= GRID_AREA[3])) | measured
    if RECON_REGION == "scan_areas" and any(s.get("disk") is not None for s in SCANS):
        return scan_disk_mask(scans) | measured
    if RECON_REGION == "scan_areas":  # no spirals/circles in this run (GP path, edge explorer)
        pass  # falls through to the near-measured mask below
    if RECON_REGION in ("near_measured", "scan_areas"):
        from scipy.ndimage import distance_transform_edt
        px = (BOUNDS[1] - BOUNDS[0]) / (N_PIXELS - 1), (BOUNDS[3] - BOUNDS[2]) / (N_PIXELS - 1)
        return distance_transform_edt(~measured, sampling=(px[1], px[0])) <= NEAR_MEASURED_V
    raise ValueError(f"Unknown RECON_REGION {RECON_REGION!r}")


def reconstruct_region(binned, scans, pts=None):
    """reconstruct_binned, blanked (NaN) outside region_mask."""
    recon = reconstruct_binned(binned, BOUNDS, pts)
    return np.where(region_mask(binned, scans), recon, np.nan)


GRID_AREA = tuple(float(v) for v in d["grid_area"]) if "grid_area" in d.files else None
if RECON_REGION == "scan_areas" and GRID_AREA is None and not any(s.get("disk") is not None for s in SCANS):
    print("No spirals / circles in this run -- 'scan_areas' uses the 'near_measured' region instead")
print(f"RECON_MODE={RECON_MODE!r}, RECON_REGION={RECON_REGION!r}, {N_PIXELS}x{N_PIXELS} px, "
      f"window X {BOUNDS[0]:.3f}..{BOUNDS[1]:.3f} V, Y {BOUNDS[2]:.3f}..{BOUNDS[3]:.3f} V")

## 4. Final reconstruction

Re-run this cell (after changing section 3) as many times as you like -- nothing
here touches the original file or the instrument.

In [ ]:
if MODALITY in ("edge_follow", "edge_explorer", "grid_refine"):
    t_rec = time.time()
    BINNED, _ = bin_image(points[:, :2], points[:, 2], N_PIXELS, BOUNDS)
    RECON = reconstruct_region(BINNED, SCANS, points)
    print(f"{RECON_MODE} reconstruction in {time.time() - t_rec:.1f}s -- "
          f"{np.isfinite(BINNED).mean():.1%} of pixels measured, "
          f"{np.isfinite(RECON).mean():.1%} reconstructed ({RECON_REGION})")

    vmin, vmax = float(np.nanmin(BINNED)), float(np.nanmax(BINNED))
    panels = [(BINNED, BOUNDS, "Binned AI0 (measured pixels)")]
    if "recon_Z" in d.files:
        if "recon_bounds" in d.files:
            saved_b = tuple(float(v) for v in d["recon_bounds"])
        else:
            saved_b = (float(d["recon_X"].min()), float(d["recon_X"].max()),
                       float(d["recon_Y"].min()), float(d["recon_Y"].max()))
        saved_mode = str(d["recon_mode"]) if "recon_mode" in d.files else "interpolated"
        panels.append((d["recon_Z"], saved_b, f"Saved at scan time ({RECON_TITLES.get(saved_mode, saved_mode)})"))
    _where = {"scan_areas": ", scan areas only", "near_measured": ", near measured only", "full": ""}[RECON_REGION]
    panels.append((RECON, BOUNDS, f"Recomputed ({RECON_TITLES[RECON_MODE]}{_where})"))

    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 4.6), layout="constrained")
    for ax, (img, b, title) in zip(np.atleast_1d(axes), panels):
        im = ax.imshow(img, origin="lower", extent=b, cmap="copper", vmin=vmin, vmax=vmax,
                       interpolation="nearest")
        ax.set_xlim(BOUNDS[0], BOUNDS[1]); ax.set_ylim(BOUNDS[2], BOUNDS[3])
        ax.set_title(title); ax.set_xlabel("X (V)"); ax.set_ylabel("Y (V)")
        fig.colorbar(im, ax=ax, label=Z_LABEL)
    plt.show()

    _Xg, _Yg = np.meshgrid(np.linspace(BOUNDS[0], BOUNDS[1], N_PIXELS), np.linspace(BOUNDS[2], BOUNDS[3], N_PIXELS))
    NEW_RECON = dict(recon_X=_Xg, recon_Y=_Yg, recon_Z=RECON, binned_Z=BINNED,
                     recon_bounds=np.array(BOUNDS), recon_mode=RECON_MODE,
                     recon_region=RECON_REGION, recon_mask=np.isfinite(RECON))

else:  # "gp_path"
    EXTENT_V = 0.45       # half-width of the GP reconstruction grid, in bias volts
    GRID_N = 300
    MAX_POINTS = 20000
    Xg, Yg = build_grid(EXTENT_V, n=GRID_N)
    from sklearn.gaussian_process import GaussianProcessRegressor
    from sklearn.gaussian_process.kernels import RBF, WhiteKernel, ConstantKernel as C

    # --- GP knobs -- re-fit from scratch on the saved points, independent of
    # whatever the live run's GPPathConfig used ---
    LENGTH_SCALE = 0.05
    NOISE_LEVEL = 1e-3
    MAX_TRAIN_POINTS = 3000  # GP fit is O(n^2) memory / O(n^3) time -- keep this modest

    xy, z = subsample_for_reconstruction(points[:, :2], points[:, 2], MAX_TRAIN_POINTS)
    kernel = C(1.0, (1e-2, 1e2)) * RBF(length_scale=LENGTH_SCALE) + WhiteKernel(noise_level=NOISE_LEVEL)
    gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=2, alpha=1e-8)
    gp.fit(xy, z)
    mu_new, sigma_new = gp.predict(np.column_stack([Xg.ravel(), Yg.ravel()]), return_std=True)
    mu_new = mu_new.reshape(Xg.shape)
    sigma_new = sigma_new.reshape(Xg.shape)

    fig, axes = plt.subplots(2, 2, figsize=(12, 11))
    axes[0, 0].pcolormesh(d["recon_X"], d["recon_Y"], d["recon_mean"], cmap="copper", shading="auto")
    axes[0, 0].set_title("Mean -- saved at scan time"); axes[0, 0].set_aspect("equal")
    axes[0, 1].pcolormesh(Xg, Yg, mu_new, cmap="copper", shading="auto")
    axes[0, 1].set_title(f"Mean -- recomputed (length_scale={LENGTH_SCALE})"); axes[0, 1].set_aspect("equal")
    axes[1, 0].pcolormesh(d["recon_X"], d["recon_Y"], d["recon_sigma"], cmap="hot", shading="auto")
    axes[1, 0].set_title("Uncertainty -- saved"); axes[1, 0].set_aspect("equal")
    axes[1, 1].pcolormesh(Xg, Yg, sigma_new, cmap="hot", shading="auto")
    axes[1, 1].set_title("Uncertainty -- recomputed"); axes[1, 1].set_aspect("equal")
    plt.tight_layout()
    plt.show()

    NEW_RECON = dict(recon_X=Xg, recon_Y=Yg, recon_mean=mu_new, recon_sigma=sigma_new)


### 4a. In micrometres (runs with position sensors)

With `cfg.position_sensors` on and calibrated, the run's positions are the **measured** ones
(sensors, in the calibrated drive-volt frame) -- so section 4's images are already corrected
for the piezo's errors. This shows them with axes in micrometres along the sensors' axes
(`um_per_V`, `um_at_0V` in the file: um = um_per_V x V + um_at_0V), plus the calibration
measured in the setup.

In [ ]:
if "sensor_calibration_json" in d.files and MODALITY != "gp_path":
    import json as _json
    from selfdriving_afm.sensors import SensorCalibration
    from selfdriving_afm.viz import um_axis_ticks
    _c = _json.loads(str(d["sensor_calibration_json"]))
    SENSOR_CAL = SensorCalibration.from_dict(_c)
    r = _c.get("report", {})
    upv = SENSOR_CAL.um_per_V
    print(f"calibration: X {upv[0]:+.4f} um per drive V, Y {upv[1]:+.4f} um per drive V"
          + (f"; straight-line residual {r['residual_nm'][0]:.1f} / {r['residual_nm'][1]:.1f} nm, sensor noise "
             f"{r['noise_nm'][0]:.2f} / {r['noise_nm'][1]:.2f} nm" if r else ""))
    _xu, _yu = SENSOR_CAL.um(np.array(BOUNDS[:2]), np.array(BOUNDS[2:]))
    print(f"image: {abs(_xu[1] - _xu[0]):.3g} x {abs(_yu[1] - _yu[0]):.3g} um, pixel "
          f"{abs(_xu[1] - _xu[0]) / (N_PIXELS - 1) * 1e3:.3g} x {abs(_yu[1] - _yu[0]) / (N_PIXELS - 1) * 1e3:.3g} nm")
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), layout="constrained")
    for ax, img, title in zip(axes, (BINNED, RECON), ("Binned AI0 (measured positions)",
                                                       f"{RECON_TITLES[RECON_MODE]} reconstruction")):
        im = ax.imshow(img, origin="lower", extent=BOUNDS, cmap="copper", vmin=vmin, vmax=vmax, interpolation="nearest")
        ax.set_xlim(BOUNDS[0], BOUNDS[1]); ax.set_ylim(BOUNDS[2], BOUNDS[3])
        um_axis_ticks(ax, SENSOR_CAL)
        ax.set_xlabel("X (um, sensor)"); ax.set_ylabel("Y (um, sensor)"); ax.set_title(title)
        fig.colorbar(im, ax=ax, label=Z_LABEL)
    plt.show()
else:
    print("no position-sensor calibration in this file (cfg.position_sensors off, or not calibrated)")

### 4b. Reference raster (if this run has one)

`Run_SelfDriving_Scan.ipynb` section 6 can acquire, right after a run, a plain raster with
about the same number of measured pixels over the same area, saved into the same .npz
(`raster_*`). If this file has it: the raster's binned image and its reconstruction next to
this notebook's reconstruction of the self-driving run, on one colour scale (raw AI0, as
recomputed here).

The raster is reconstructed with `RASTER_RECON` -- by default `"raster_columns"`:
**PCHIP across the scan lines** (along each pixel column for lines along x, each pixel
row for lines along y -- detected from the data), the same as `Simulation.ipynb`'s classic
raster. That's the best method for raster data (compressed sensing's DCT prior aliases
on regularly skipped lines). `RASTER_ALSO_RECON_MODE = True` adds a panel with section 3's
`RECON_MODE` too, for comparison.

With the position sensors on, the raster is shown **at its measured positions** (`RASTER_POSITIONS
= "auto"`): along each line where the sensors put the tip, each line at its measured place --
no time-shift fit involved (Z and the sensors come from the same AI record). Axes in um if
section 4a ran. `"commanded"` shows the commanded-position version (time-shift fitted).

In [ ]:
RASTER_RECON = "raster_columns"   # the raster's reconstruction: "raster_columns" = column-wise PCHIP
                                  #   between the lines (default, as in Simulation.ipynb), or "linear",
                                  #   "fast", "compressed_sensing", "gp"
RASTER_ALSO_RECON_MODE = False    # True: also show it reconstructed with section 3's RECON_MODE
RASTER_POINTS_PER_LINE = 1000     # the raster binned with this many points ALONG each line (its lines
                                  #   are sampled far finer than the run's grid); limited to what the
                                  #   saved samples support. None = the run's grid (200)

RASTER_POSITIONS = "auto"         # "auto": at the measured positions (sensors) if the raster recorded them,
                                  #   else commanded; or "sensor" / "commanded"

RECON_TITLES.setdefault("raster_columns", "Column-wise PCHIP")
if "raster_binned_Z" in d.files and MODALITY != "gp_path":
    _rb = tuple(float(v) for v in d["raster_bounds"])
    from selfdriving_afm import reference_raster
    # time-shift corrected (Z vs path on PyScanner's sinusoidal fast raster -- files saved
    # before the correction are corrected here, from their saved samples)
    RASTER = reference_raster.load(d, points_per_line=RASTER_POINTS_PER_LINE, positions=RASTER_POSITIONS)
    RASTER_BINNED = RASTER["binned"]
    print(f"Reference raster at the {'MEASURED (sensors)' if RASTER['positions'] == 'sensor' else 'commanded'} positions"
          + ("" if RASTER["positions"] == "sensor" or "raster_xs" not in d.files else " (the file also has the measured ones)"))
    if RASTER["delay_s"] and RASTER["positions"] != "sensor":
        print(f"Reference raster: Z placed {RASTER['delay_s'] * 1e3:+.1f} ms along the path "
              f"(trace and retrace agree)")
    RASTER_RECON_IMG = inpaint_binned(RASTER_BINNED, _rb, _recon_cfg, mode=RASTER_RECON)
    print(f"Reference raster: {int(d['raster_n_lines'])} lines {float(d['raster_spacing_V']):.3g} V apart, "
          f"{int(d['raster_pixels']):,} pixels measured (the self-driving run: {int(d['raster_run_pixels']):,}), "
          f"{float(d['raster_path_V']):.0f} V of path, ~{float(d['raster_scan_time_s']):.0f} s of waveform "
          f"({float(d['raster_wall_time_s']):.0f} s wall time); reconstructed with {RECON_TITLES.get(RASTER_RECON, RASTER_RECON)}, "
          f"{RASTER['points_per_line']} points per line")
    _panels = [(RECON, BOUNDS, f"Self-driving run ({RECON_TITLES[RECON_MODE]})\n"
                               f"{np.isfinite(BINNED).sum():,} pixels measured"),
               (reference_raster.for_display(RASTER_BINNED), _rb, f"Reference raster: binned{', measured positions' if RASTER['positions'] == 'sensor' else ''}\n{int(d['raster_n_lines'])} lines x "
                                    f"{RASTER['points_per_line']} points"),
               (RASTER_RECON_IMG, _rb, f"Reference raster: {RECON_TITLES.get(RASTER_RECON, RASTER_RECON)}")]
    if RASTER_ALSO_RECON_MODE and RECON_MODE != RASTER_RECON and RECON_MODE != "interpolated":
        _panels.append((inpaint_binned(RASTER_BINNED, _rb, _recon_cfg), _rb,
                        f"Reference raster: {RECON_TITLES[RECON_MODE]}"))
    _vmin, _vmax = np.nanpercentile(RASTER_BINNED, [1, 99])
    fig, axes = plt.subplots(1, len(_panels), figsize=(4.8 * len(_panels), 4.6), layout="constrained")
    for ax, (img, b, title) in zip(axes, _panels):
        im = ax.imshow(img, origin="lower", extent=b, cmap="copper", vmin=_vmin, vmax=_vmax, interpolation="nearest")
        ax.set_xlim(BOUNDS[0], BOUNDS[1]); ax.set_ylim(BOUNDS[2], BOUNDS[3])
        ax.set_title(title, fontsize=9); ax.set_xlabel("X (V)")
        if "SENSOR_CAL" in globals():      # section 4a: axes in um along the sensors
            from selfdriving_afm.viz import um_axis_ticks
            um_axis_ticks(ax, SENSOR_CAL); ax.set_xlabel("X (um, sensor)"); ax.set_ylabel("Y (um, sensor)")
    fig.colorbar(im, ax=list(axes), label=Z_LABEL, shrink=0.85)
    plt.show()
else:
    print("No reference raster in this file (Run_SelfDriving_Scan.ipynb section 6 acquires one)")


## 5. Animation of the run

Replays the saved run scan by scan -- same three columns as the live view:
**scan path** (blue = spiral search, green = edge trace, dashed gray = moves between
scans) | **binned AI0** | **reconstruction** (section 3's `RECON_MODE` and
parameters). One frame per scan; the title says which scan it is. All panels use
the final window (`BOUNDS`), so the axes stay still while the data fills in.

Saved next to the `.npz` as `<run name>_animation.gif` (or `.mp4`). Long runs:
raise `FRAME_EVERY` (fewer frames) and/or `RECON_EVERY` (recompute the
reconstruction less often -- GP is ~1 s per frame). With `RECON_REGION` other than
`"full"`, the reconstructed area grows frame by frame with the scanned area.

The saved file doesn't record which feature a circle belongs to, so frames say
"Edge-trace circle #N" rather than "feature X, circle N".

**Reference raster (second row).** If this run has one (Run notebook section 6), the
animation gets a second row: the raster's lines | binned AI0 | reconstruction (section 4b's
`RASTER_RECON`, column-wise PCHIP by default). With `RASTER_ANIM_MATCH = "pixels"` the raster
builds up as it would on the microscope: at its final line spacing, line 1, line 2, ... (the
last one partial, in its scan direction; the reconstruction only up to that line), with as
many measured pixels in every frame as the self-driving scan has so far -- so the two rows
always compare equal budgets (the raster was planned to match the whole run, so the last
frame is all or nearly all of it). `"full"` shows the whole raster throughout.
`RASTER_IN_ANIMATION = False` turns the row off.

In [ ]:
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter, writers
from IPython.display import Image as _IPImage, Video as _IPVideo, display

ANIM_FORMAT = "gif"    # "gif" (always works) or "mp4" (needs ffmpeg installed)
ANIM_FPS = 4           # frames per second
FRAME_EVERY = None     # one frame every N scans; None = automatic (about MAX_FRAMES frames)
MAX_FRAMES = 120       # used when FRAME_EVERY is None
RECON_EVERY = 1        # recompute the reconstruction every N frames (reused in between)
ANIM_DPI = 80          # resolution -- higher = sharper but bigger file
ANIM_FIGSIZE = (13.5, 4.4)
SHOW_IN_NOTEBOOK = True
# second row: the reference raster (if this run has one -- Run notebook section 6)
RASTER_IN_ANIMATION = True   # False = one row only, even if the file has a raster
# 4th column: uncertainty of every row's reconstruction -- 0% on measured pixels, rising with
# the distance to the nearest measured one, 100% at UNCERTAINTY_SCALE_V or more (a gap the
# smallest feature fits in) -- reconstruction.uncertainty_map
UNCERTAINTY_COLUMN = True
UNCERTAINTY_SCALE_V = None   # None = the run's smallest feature (its priors), else 5% of the width
RASTER_ANIM_MATCH = "pixels"  # "pixels": the raster as it was scanned -- at its final line spacing, line 1,
                              #   line 2, ... (the last one partial), with as many measured pixels in every
                              #   frame as the self-driving scan has so far (equal budgets);
                              #   "full": the whole reference raster in every frame

# frames = (scan, last sample shown) -- normally one per scan (SCANS: the run split back
# into its scans, section 1). A run of few, long scans (grid+refine sends one waveform per
# level) is split into several frames per scan instead, in proportion to their length.
scans, offsets = SCANS, OFFSETS
if FRAME_EVERY is None and len(scans) < MAX_FRAMES / 2:
    _starts = [int(offsets[s["steps"][0]]) for s in scans]
    _total = max(scans[-1]["end_sample"] - _starts[0], 1)
    frames = []
    for i, (a, s) in enumerate(zip(_starts, scans)):
        b = s["end_sample"]
        k = max(1, int(round(MAX_FRAMES * (b - a) / _total)))
        frames += [(i, int(a + (b - a) * (j + 1) / k)) for j in range(k)]
    _every_txt = "long scans split into several frames"
else:
    _every = FRAME_EVERY if FRAME_EVERY else max(1, int(np.ceil(len(scans) / MAX_FRAMES)))
    _idx = list(range(_every - 1, len(scans), _every))
    if not _idx or _idx[-1] != len(scans) - 1:
        _idx.append(len(scans) - 1)
    frames = [(i, scans[i]["end_sample"]) for i in _idx]
    _every_txt = f"one every {_every}"
print(f"{len(scans)} scans -> {len(frames)} frames ({_every_txt}); reconstruction: {RECON_MODE}, "
      f"region: {RECON_REGION}")

# --- figure: path | binned AI0 | reconstruction, fixed window -----------------------
_x0, _x1, _y0, _y1 = BOUNDS
_n = N_PIXELS
# same pixel convention as bin_image / PyScanner's image_AI0: round(t * (n - 1))
_col = np.clip(np.round((points[:, 0] - _x0) / (_x1 - _x0) * (_n - 1)), 0, _n - 1).astype(np.int64)
_row = np.clip(np.round((points[:, 1] - _y0) / (_y1 - _y0) * (_n - 1)), 0, _n - 1).astype(np.int64)
_flat = _row * _n + _col
del _col, _row  # can be large (one entry per sample)
_inside = ((points[:, 0] >= _x0) & (points[:, 0] <= _x1) & (points[:, 1] >= _y0) & (points[:, 1] <= _y1))
_sums = np.zeros(_n * _n)
_counts = np.zeros(_n * _n)

_final_binned, _ = bin_image(points[:, :2], points[:, 2], _n, BOUNDS)
vmin, vmax = float(np.nanmin(_final_binned)), float(np.nanmax(_final_binned))

_HAS_RASTER = RASTER_IN_ANIMATION and "raster_binned_Z" in d.files and MODALITY != "gp_path"
_NC = 4 if UNCERTAINTY_COLUMN else 3
_FS = (ANIM_FIGSIZE[0] * _NC / 3, ANIM_FIGSIZE[1])   # ANIM_FIGSIZE is for 3 columns
axU = axU2 = None
if _HAS_RASTER:
    fig, _axs = plt.subplots(2, _NC, figsize=(_FS[0], 2 * _FS[1]), layout="constrained")
    (axP, axB, axR), (axP2, axB2, axR2) = _axs[0][:3], _axs[1][:3]
    if UNCERTAINTY_COLUMN:
        axU, axU2 = _axs[0][3], _axs[1][3]
else:
    fig, _axs = plt.subplots(1, _NC, figsize=_FS, layout="constrained")
    axP, axB, axR = _axs[:3]
    if UNCERTAINTY_COLUMN:
        axU = _axs[3]
plt.close(fig)  # rendered into the file, not shown as a static figure

from selfdriving_afm.reconstruction import uncertainty_map
import json as _json
_UNC_SCALE = UNCERTAINTY_SCALE_V
if _UNC_SCALE is None and "priors_json" in d.files:
    _UNC_SCALE = _json.loads(str(d["priors_json"])).get("feature_min_V")
_UNC_WHAT = f"{_UNC_SCALE:.3g} V" if _UNC_SCALE else "5% of the width"
if _UNC_SCALE and "priors_json" in d.files and UNCERTAINTY_SCALE_V is None:
    _e = _json.loads(str(d["priors_json"])).get("entered")
    if _e:                                   # the priors were given in nm
        _UNC_WHAT = f"{_e['feature_min_nm']:g} nm"


def _setup_unc(ax, title):
    im = ax.imshow(np.full((2, 2), 100.0), origin="lower", extent=BOUNDS, cmap="magma", vmin=0, vmax=100,
                   interpolation="nearest")
    fig.colorbar(im, ax=ax, label="uncertainty (%)")
    ax.set_xlim(_x0, _x1); ax.set_ylim(_y0, _y1); ax.set_aspect("equal")
    ax.set_xlabel("X (V)"); ax.set_ylabel("Y (V)"); ax.set_title(title)
    return im


def _draw_unc(ax, im, binned, bounds, who):
    u = 100.0 * uncertainty_map(binned, bounds, _UNC_SCALE)
    im.set_data(u)
    im.set_extent(bounds)
    ax.set_title(f"{who}uncertainty (100% = {_UNC_WHAT} from data)\nmean {u.mean():.0f}%, "
                 f"{np.mean(u >= 99.9):.0%} of pixels fully unknown", fontsize=9)


if axU is not None:
    imU = _setup_unc(axU, "Uncertainty")
_blank = np.full((_n, _n), np.nan)
imB = axB.imshow(_blank, origin="lower", extent=BOUNDS, cmap="copper", vmin=vmin, vmax=vmax, interpolation="nearest")
imR = axR.imshow(_blank, origin="lower", extent=BOUNDS, cmap="copper", vmin=vmin, vmax=vmax, interpolation="nearest")
fig.colorbar(imB, ax=axB, label=Z_LABEL)
fig.colorbar(imR, ax=axR, label=Z_LABEL)
for ax, title in ((axP, "Scan path"), (axB, "Binned AI0"), (axR, f"{RECON_TITLES[RECON_MODE]} reconstruction"
                                                                        f"{' (scanned areas)' if RECON_REGION != 'full' else ''}")):
    ax.set_xlim(_x0, _x1); ax.set_ylim(_y0, _y1); ax.set_aspect("equal")
    ax.set_xlabel("X (V)"); ax.set_ylabel("Y (V)"); ax.set_title(title)
axP.grid(True, alpha=0.3)
for kind in ("spiral", "circle", "line", "explore", "grid_sparse", "refine", "connector"):  # legend entries only
    if kind in KINDS:
        axP.plot([], [], **_KIND_STYLE[kind])
axP.legend(loc="upper right", fontsize=7)
tip, = axP.plot([], [], "ro", markersize=6, zorder=5)
status = fig.suptitle("", fontsize=11, fontweight="bold")

if _HAS_RASTER:
    # --- second row: the reference raster, on its own grid (same area and size as the run's) ---
    from selfdriving_afm.reconstruction import raster_lines_along
    from selfdriving_afm import reference_raster
    _RASTER_RECON = globals().get("RASTER_RECON", "raster_columns")   # section 4b's choice (PCHIP by default)
    _rb = tuple(float(v) for v in d["raster_bounds"])
    if "RASTER_BINNED" in globals():      # section 4b's (time-shift corrected)
        _rfull = np.asarray(RASTER_BINNED)
    else:
        from selfdriving_afm import reference_raster
        _rfull = reference_raster.load(d, points_per_line=reference_raster.POINTS_PER_LINE,
                                       positions=globals().get("RASTER_POSITIONS", "auto"))["binned"]
    _rn = _rfull.shape[0]          # points per line (the raster's own, finer grid)
    _rrun = int(d["raster_binned_Z"].shape[0])   # the run's grid: the pixel budget is counted on it
    _r_along = raster_lines_along(_rfull)
    _rM = np.isfinite(_rfull)
    # the raster's measured lines (pixel rows for lines along x, columns for lines along y), in order
    _r_lines = np.nonzero((_rM.mean(axis=1) if _r_along == "x" else _rM.mean(axis=0)) > 0.5)[0]
    _rblank = np.full((_rn, _rn), np.nan)
    imB2 = axB2.imshow(_rblank, origin="lower", extent=_rb, cmap="copper", vmin=vmin, vmax=vmax, interpolation="nearest")
    imR2 = axR2.imshow(_rblank, origin="lower", extent=_rb, cmap="copper", vmin=vmin, vmax=vmax, interpolation="nearest")
    fig.colorbar(imB2, ax=axB2, label=Z_LABEL)
    fig.colorbar(imR2, ax=axR2, label=Z_LABEL)
    for ax, title in ((axP2, "Reference raster: lines"), (axB2, "Reference raster: binned AI0"),
                      (axR2, f"Reference raster: {RECON_TITLES.get(_RASTER_RECON, 'Column-wise PCHIP')}")):
        ax.set_xlim(_x0, _x1); ax.set_ylim(_y0, _y1); ax.set_aspect("equal")
        ax.set_xlabel("X (V)"); ax.set_ylabel("Y (V)"); ax.set_title(title)
    axP2.grid(True, alpha=0.3)
    if axU2 is not None:
        imU2 = _setup_unc(axU2, "Reference raster: uncertainty")
    _rpath, = axP2.plot([], [], "-", color="tab:red", lw=0.8)
    _rtip, = axP2.plot([], [], "o", color="tab:red", markersize=6, zorder=5)
    _rcoord = np.linspace(_rb[2], _rb[3], _rn) if _r_along == "x" else np.linspace(_rb[0], _rb[1], _rn)
    _racross = (_rb[0], _rb[1]) if _r_along == "x" else (_rb[2], _rb[3])

    _r_fast = str(d["raster_generator"]) == "pyscanner_fast_raster" if "raster_generator" in d.files else False

    def _line_dir(j):
        """+1: line j is scanned towards the high fast-axis end first. PyScanner's fast raster:
        lines along x start at low x, lines along y (rotated 90 deg clockwise) at high y; the
        older user-defined serpentine alternates, starting at low x / low y."""
        if _r_fast:
            return 1 if _r_along == "x" else -1
        return 1 if j % 2 == 0 else -1

    def raster_frame(n_measured):
        """The reference raster as scanned in situ: its lines in scan order at the final
        spacing, lines 1..m complete and line m+1 partial, n_measured pixels in all (or the
        whole raster, RASTER_ANIM_MATCH = "full"). Returns (binned, recon mask, lines shown,
        last length fraction)."""
        k = len(_r_lines)
        if RASTER_ANIM_MATCH == "full" or k == 0:
            return _rfull, np.ones_like(_rfull, dtype=bool), _r_lines, 1.0
        n_px = int(min(max(n_measured, 0), k * _rn))
        m, part = divmod(n_px, _rn)
        sel = _r_lines[:m + (1 if part and m < k else 0)]
        img = np.full((_rn, _rn), np.nan)
        shown = np.zeros((_rn, _rn), dtype=bool)   # where the recon is drawn: up to the current line
        frac = 1.0
        for j, r in enumerate(sel):
            take = part if (j == m and part) else _rn
            frac = take / _rn
            lo, hi = (0, take) if _line_dir(j) > 0 else (_rn - take, _rn)
            prev = _r_lines[j - 1] + 1 if j > 0 else 0
            if _r_along == "x":
                img[r, lo:hi] = _rfull[r, lo:hi]
                shown[prev:r + 1, lo:hi] = True
                if j < m:
                    shown[prev:r + 1, :] = True
            else:
                img[lo:hi, r] = _rfull[lo:hi, r]
                shown[lo:hi, prev:r + 1] = True
                if j < m:
                    shown[:, prev:r + 1] = True
        return img, shown, sel, frac

    def draw_raster_row(n_measured, do_recon):
        img, shown, sel, frac = raster_frame(n_measured)
        xs, ys = [], []
        for j, r in enumerate(sel):
            a0, a1 = _racross if _line_dir(j) > 0 else _racross[::-1]
            a1 = a0 + (a1 - a0) * (frac if j == len(sel) - 1 else 1.0)
            c = _rcoord[r]
            if _r_along == "x":
                xs += [a0, a1, np.nan]; ys += [c, c, np.nan]
            else:
                xs += [c, c, np.nan]; ys += [a0, a1, np.nan]
        _rpath.set_data(xs, ys)
        if len(sel):
            _rtip.set_data([xs[-2]], [ys[-2]])
        imB2.set_data(reference_raster.for_display(img))   # lines drawn as bands (display only)
        if axU2 is not None:
            _draw_unc(axU2, imU2, img, _rb, "Reference raster: ")   # from the lines themselves
        if do_recon and np.isfinite(img).sum() >= 3:
            rec = np.array(inpaint_binned(img, _rb, _recon_cfg, mode=_RASTER_RECON), dtype=float)
            rec[~shown] = np.nan     # nothing beyond the line being scanned
            imR2.set_data(rec)
        return int(np.isfinite(img).sum()), len(sel)

x_cmd, y_cmd = d["x_cmd"], d["y_cmd"]
_state = dict(sample=0)


def draw_frame(k):
    si, b = frames[k]
    s = scans[si]
    a = _state["sample"]
    # path: the samples since the last frame, coloured by the waveform they belong to
    # (strided, for drawing only)
    for i in range(int(np.searchsorted(offsets, a, side="right")) - 1, len(offsets) - 1):
        lo, hi = max(offsets[i], a), min(offsets[i + 1], b)
        if lo >= b:
            break
        if hi <= lo:
            continue
        stride = max(1, (hi - lo) // 1500)
        style = {key: v for key, v in _KIND_STYLE.get(KINDS[i], {}).items() if key != "label"}
        axP.plot(x_cmd[lo:hi:stride], y_cmd[lo:hi:stride], **style)
    tip.set_data([x_cmd[b - 1]], [y_cmd[b - 1]])
    # binned AI0: add the new samples
    sel = _inside[a:b]
    np.add(_counts, np.bincount(_flat[a:b][sel], minlength=_n * _n), out=_counts)  # in place
    np.add(_sums, np.bincount(_flat[a:b][sel], weights=points[a:b, 2][sel], minlength=_n * _n), out=_sums)
    _state["sample"] = b
    with np.errstate(invalid="ignore", divide="ignore"):
        binned = np.where(_counts > 0, _sums / _counts, np.nan).reshape(_n, _n)
    imB.set_data(binned)
    if axU is not None:
        _draw_unc(axU, imU, binned, BOUNDS, "Self-driving: ")
    if (k % RECON_EVERY == 0 or k == len(frames) - 1) and np.isfinite(binned).sum() >= 10:
        # only inside what's been scanned SO FAR -- the region grows with the scan
        imR.set_data(reconstruct_region(binned, scans[:si + 1], points[:b]))
    pieces = f" ({s['pieces']} pieces)" if s["pieces"] > 1 else ""
    text = f"{s['label']}{pieces}  --  scan {si + 1}/{len(scans)}, {b:,} points"
    if _HAS_RASTER:
        n_meas = int(np.isfinite(binned).sum())
        # same budget: the raster as far as n_meas pixels of the run's grid take it (its
        # lines, at _rn points each instead of _rrun)
        r_px, r_lines = draw_raster_row(int(round(n_meas * _rn / _rrun)), k % RECON_EVERY == 0 or k == len(frames) - 1)
        text += (f"\nself-driving: {n_meas:,} pixels measured  |  raster: {r_lines} line(s) "
                 f"({_rn} points per line)")
    status.set_text(text)
    return []


ext = "mp4" if ANIM_FORMAT == "mp4" else "gif"
if ext == "mp4" and not writers.is_available("ffmpeg"):
    print("ffmpeg not found -- saving a GIF instead")
    ext = "gif"
anim_path = pathlib.Path(SCAN_FILE).with_name(pathlib.Path(SCAN_FILE).stem + f"_animation.{ext}")
writer = FFMpegWriter(fps=ANIM_FPS) if ext == "mp4" else PillowWriter(fps=ANIM_FPS)


def _progress(i, total):
    if (i + 1) % 10 == 0 or i + 1 == total:
        print(f"  rendered frame {i + 1}/{total}")


t_anim = time.time()
anim = FuncAnimation(fig, draw_frame, frames=len(frames), blit=False, repeat=False)
anim.save(str(anim_path), writer=writer, dpi=ANIM_DPI, progress_callback=_progress)
print(f"Saved {anim_path} ({anim_path.stat().st_size / 1e6:.1f} MB, {len(frames)} frames) "
      f"in {time.time() - t_anim:.0f}s")

if SHOW_IN_NOTEBOOK:
    display(_IPVideo(str(anim_path), embed=True) if ext == "mp4" else _IPImage(filename=str(anim_path)))

### 5b. Open loop vs closed loop: self-driving and raster, side by side

One animation, up to **4 rows**, all advancing together (`selfdriving_afm/compare_animation.py`):
1. self-driving run at the **commanded** positions (open loop, drive volts);
2. self-driving run at the **measured** positions (closed loop: the X/Y sensors, axes in um);
3. reference raster at the commanded positions;
4. reference raster at the measured positions.

Each row: path | binned | reconstruction | uncertainty, like the run's live view. Frames follow
the self-driving run's waveforms; a raster row shows the raster as far as it takes to measure
**as many pixels** (of the run's grid) as its self-driving partner row has so far -- equal
budgets, in scan order. Rows the file doesn't have (no sensors, no raster) are left out.
Z in the file's units; the fitted substrate removed if the run's reconstruction was flattened.

In [ ]:
import json
from IPython.display import Image as _IPImage, Video as _IPVideo, display
from selfdriving_afm import compare_animation

CMP_FORMAT = "gif"        # "gif" or "mp4" (needs ffmpeg)
CMP_MAX_FRAMES = 60       # frames (one per ~1/60 of the run's waveforms)
CMP_FPS = 4
CMP_DPI = 70
CMP_RECON_EVERY = 2       # reconstruct every N frames (CS / GP are the slow part)
CMP_ROWS = ("run_cmd", "run_sensor", "raster_cmd", "raster_sensor")
SHOW_CMP = True

_cmp_path = pathlib.Path(SCAN_FILE).with_name(pathlib.Path(SCAN_FILE).stem + f"_open_vs_closed.{CMP_FORMAT}")
CMP_ANIM = compare_animation.animate(
    d, _cmp_path, rows=CMP_ROWS, bounds=BOUNDS, n_pixels=N_PIXELS, recon_cfg=_recon_cfg, recon_mode=RECON_MODE,
    raster_recon=globals().get("RASTER_RECON", "raster_columns"), max_frames=CMP_MAX_FRAMES, fps=CMP_FPS,
    dpi=CMP_DPI, recon_every=CMP_RECON_EVERY, length_unit="um",
    title=f"{MODALITY}: commanded (open loop) vs measured (closed loop) positions")
if SHOW_CMP and CMP_ANIM is not None:
    display(_IPVideo(str(CMP_ANIM), embed=True) if CMP_ANIM.suffix == ".mp4" else _IPImage(filename=str(CMP_ANIM)))

### 5c. Closed loop in nm: self-driving vs raster

The same, with only the **measured** (sensor) rows -- the self-driving run and the reference
raster (if the file has it) -- and the axes in **nm** along the sensors. Needs the position
sensors (and a calibration) in the file.

In [ ]:
if "x_sensor" in d.files and "sensor_calibration_json" in d.files:
    _nm_path = pathlib.Path(SCAN_FILE).with_name(pathlib.Path(SCAN_FILE).stem + f"_closed_loop_nm.{CMP_FORMAT}")
    _e = json.loads(str(d["priors_json"])).get("entered") if "priors_json" in d.files else None
    NM_ANIM = compare_animation.animate(
        d, _nm_path, rows=("run_sensor", "raster_sensor"), bounds=BOUNDS, n_pixels=N_PIXELS, recon_cfg=_recon_cfg,
        recon_mode=RECON_MODE, raster_recon=globals().get("RASTER_RECON", "raster_columns"),
        max_frames=CMP_MAX_FRAMES, fps=CMP_FPS, dpi=CMP_DPI, recon_every=CMP_RECON_EVERY, length_unit="nm",
        unc_label=(f"{_e['feature_min_nm']:g} nm" if _e else None),
        title=f"{MODALITY}: measured positions (closed loop), self-driving vs raster")
    if SHOW_CMP and NM_ANIM is not None:
        display(_IPVideo(str(NM_ANIM), embed=True) if NM_ANIM.suffix == ".mp4" else _IPImage(filename=str(NM_ANIM)))
else:
    print("No position-sensor data in this file -- no closed-loop animation")

## 6. Save the recomputed reconstruction (optional)

Writes a new `.npz` next to the original -- the original is never modified.

In [ ]:
src = pathlib.Path(SCAN_FILE)
out_path = src.with_name(src.stem + "_reanalyzed.npz")
np.savez(out_path, modality=MODALITY, **NEW_RECON)
print(f"Saved {out_path}")